# Tu primer experimento cuántico

*Usa Qiskit para calcular la energía de un sistema de Ising de dos espines.*

## Introducción

En el siguiente video, Olivia Lanes te guía por el contenido de esta lección. También puedes abrir el [video de YouTube](https://youtu.be/kniiVC538nY?si=13OJnaO5jyAUmoPQ) de esta lección en otra ventana.

▶ [Ver el video: Olivia explica cómo encontrar la energía de un Hamiltoniano de Ising de dos qubits con la primitiva Estimator de Qiskit](https://youtu.be/kniiVC538nY?si=13OJnaO5jyAUmoPQ)

A estas alturas ya ejecutaste tu primer circuito cuántico y aprendiste lo básico de la computación cuántica: cómo se representan los estados cuánticos, cómo actúan las compuertas sobre esos estados y qué papel cumplen características cuánticas como la superposición y el entrelazamiento. Ahora es momento de poner todo esto en práctica y resolver tu primer problema en un computador cuántico.

Más adelante exploraremos el panorama general de los problemas adecuados para la computación cuántica. Por ahora nos enfocaremos en un problema del área de simulación de la naturaleza: usar un computador cuántico como un sustituto más limpio y controlable de un sistema cuántico natural. De hecho, esta fue la primera aplicación que Richard Feynman imaginó para los computadores cuánticos en la década de 1980. Como dijo en una frase famosa: "La naturaleza no es clásica, maldita sea, y si quieres hacer una simulación de la naturaleza, más te vale que sea mecanocuántica...".

En esta lección seguiremos ese principio para simular la interacción entre dos espines, que puedes imaginar como pequeños imanes. Según el signo de su interacción, pueden preferir alinearse y apuntar en la misma dirección, o antialinearse y apuntar en direcciones opuestas. Nos enfocaremos en el segundo caso, porque suele dar lugar a un comportamiento más interesante y más desafiante. Una vez que entendamos este pequeño sistema de dos qubits, mostraremos cómo las mismas ideas escalan y permiten a los computadores cuánticos aprovechar su escalamiento exponencial al simular sistemas grandes de espines.

## Dos imanes que interactúan

Para este problema usaremos dos qubits, uno por cada espín de nuestro modelo. Cada espín puede apuntar hacia arriba (estado del qubit $|0\rangle$), hacia abajo (estado del qubit $|1\rangle$) o estar en una superposición de los dos estados.

Si los espines tienen una interacción *antiferromagnética*, significa que quieren antialinearse: cuando uno apunta hacia arriba, el otro quiere apuntar hacia abajo, y viceversa.

Ahora supongamos que además hay un campo magnético que apunta de izquierda a derecha en nuestro sistema. Como este campo apunta de forma transversal a la dirección habitual arriba-abajo de los espines, se le llama campo transversal. Este campo puede invertir los espines, lo que hace que la configuración de menor energía sea una superposición específica de arreglos de espines arriba y abajo, en lugar de un único patrón de espines definido.

Podemos describir todos estos efectos con un objeto matemático llamado Hamiltoniano. El Hamiltoniano nos dice la energía del sistema para un arreglo dado de espines:

$$
H = J Z_1 Z_0 + h_x (X_1 + X_0)
$$

donde $J$ es un coeficiente que controla la intensidad de la interacción entre espines y $h_x$ es un coeficiente para la intensidad del campo magnético externo. $Z_1 Z_0$ premia o penaliza a los espines según estén alineados o antialineados, y $X_0$ y $X_1$ representan el efecto del campo magnético de invertir los espines.

En física, los sistemas tienden a acomodarse en el estado de menor energía posible, llamado estado fundamental. Encontrar este estado de menor energía es un problema común, pero requiere técnicas de optimización que están fuera del alcance de esta lección.

En su lugar, nos haremos una pregunta más sencilla: si preparamos los espines en un estado particular, ¿cuál es la energía de ese estado?

Para responderla:

1. Prepararemos los espines en un estado que elijamos
2. Mediremos la energía de ese estado usando el Hamiltoniano anterior

Este es exactamente el tipo de cálculo que aparece dentro de algoritmos cuánticos más grandes, como los algoritmos variacionales, que puedes explorar en cursos posteriores.

## Implementación en Qiskit

Antes de ponernos a escribir código, necesitamos un poco de contexto. Cuando ejecutamos un circuito cuántico, siempre terminamos midiendo los qubits. Pero hay dos tipos distintos de preguntas que podemos querer hacer sobre el resultado de esa medición: a veces solo queremos saber cuál es el estado del qubit. Otras veces queremos saber, dado el estado cuántico, cuál es el valor de una cantidad física, como la energía.

En Qiskit, estos dos tipos de preguntas se resuelven con dos herramientas distintas, llamadas primitivas.

`Sampler` responde al primer tipo de pregunta. Ejecuta el circuito muchas veces y nos dice con qué frecuencia medimos cada resultado posible, como `00`, `01`, `10` o `11`. El resultado es un histograma que muestra la probabilidad de cada resultado de medición.

`Estimator` responde al segundo tipo de pregunta. En lugar de darnos un histograma, combina muchas mediciones internamente para calcular un solo número, como la energía del estado según un Hamiltoniano que le proporcionamos.

Para ayudarte a entender cuándo y por qué usaríamos cada una de estas herramientas, recorreremos dos flujos de trabajo completos (llamados "patrones de Qiskit") aplicados al mismo sistema de dos qubits.

#### El flujo de trabajo de los patrones de Qiskit

El flujo de trabajo de los patrones de Qiskit es un marco general que usamos para resolver problemas cuánticos con Qiskit. Divide una tarea de computación cuántica en cuatro pasos:

1. Mapear el problema a un modelo que se pueda representar con circuitos cuánticos
2. Optimizar el circuito para ejecutarlo en un *backend* específico
3. Ejecutar el circuito optimizado en el *backend* seleccionado
4. Post-procesar los datos de medición sin procesar

### Experimento 1: usar `Sampler` para medir el estado

#### Mapear

En general, el paso de mapeo es donde averiguamos cómo representar un problema del mundo real en términos de qubits, operadores y mediciones. En muchas aplicaciones, esta es la parte más difícil y compleja del flujo de trabajo: incluso preguntas sencillas, como "¿qué representa cada qubit?", no siempre tienen respuestas directas.

En este experimento, sin embargo, el mapeo es sencillo a propósito. Cada grado de libertad físico corresponde directamente a un solo qubit. Gracias a esta correspondencia uno a uno, el paso de mapeo se reduce a elegir el estado cuántico que queremos preparar y escribir un circuito que prepare y mida ese estado.

Aquí prepararemos un estado de Bell entrelazado, parecido al que construimos en la primera lección de este curso:

$$
\vert\Psi\rangle = \frac{1}{\sqrt{2}}(\vert 10\rangle - \vert 01\rangle)
$$

In [ ]:
# Importa las primitivas de IBM Quantum
from qiskit import QuantumCircuit

# Prepara el estado
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.x(1)
qc.z(0)

# Mide el estado
qc.measure_all()

# Dibuja el circuito
qc.draw("mpl")

#### Optimizar

Antes de ejecutar nuestro circuito en un computador cuántico (o en un simulador, si agotaste tu tiempo gratuito en computadores cuánticos reales este mes), necesitamos prepararlo para la ejecución. Este paso se llama optimización. (Nota: este uso de la palabra "optimización" puede confundir. En computación cuántica, los problemas de optimización son una clase específica de problemas. Aquí usamos optimización para describir un paso de preparación obligatorio por el que pasa todo circuito cuántico antes de poder ejecutarse de forma eficiente en el hardware).

Durante la optimización:

1. Elegimos el *backend*: un computador cuántico real o un simulador.
2. Asignamos los qubits de nuestro circuito a qubits físicos del dispositivo.
3. Reescribimos el circuito usando solo las compuertas que el computador cuántico realmente puede ejecutar.
4. Opcionalmente, aplicamos técnicas de mitigación y supresión de errores para reducir los efectos del ruido.

En Qiskit, esto lo hace automáticamente el transpilador. Una vez que eliges tu *backend*, el transpilador hace todo el trabajo para dejar tu circuito listo para la ejecución, así que no tienes que ajustar a mano las compuertas ni la asignación de qubits. El transpilador también ofrece distintos niveles de optimización, que pueden ayudar a reducir errores si hace falta. La optimización se hace en etapas llamadas "pases" (*passes*). Por eso, en el código de abajo la optimización la maneja el `pass_manager`. Para aprender más sobre los errores y su mitigación, consulta el curso de Olivia Lanes [Quantum Computing in Practice](https://quantum.cloud.ibm.com/learning/es/courses/quantum-computing-in-practice).

In [ ]:
# Carga el servicio IBM Quantum Compute
from qiskit_ibm_runtime import QiskitRuntimeService

## Carga el servicio IBM Quantum Compute
# QiskitRuntimeService.save_account(
#    channel="ibm_quantum_platform",
#    token="YOUR_TOKEN_HERE",
#    overwrite=True,
#    set_as_default=True,
# )
# service = QiskitRuntimeService(channel="ibm_quantum_platform")


# O carga las credenciales guardadas
service = QiskitRuntimeService()

# Usa el backend menos ocupado, o descomenta la carga de un backend específico como "ibm_brisbane".
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=127)
# backend = service.backend("ibm_brisbane")
print(backend.name)

<div class="alert alert-block alert-info">
<b>Nota de la traducción: alternativa sin cuenta</b>

La celda anterior necesita credenciales guardadas de IBM Quantum. Si no las tienes, ejecuta la celda de abajo <b>en lugar de</b> la anterior: carga una copia local (*fake backend*) de un procesador real de IBM, con sus compuertas, su conectividad y su modelo de ruido. Los dos experimentos funcionan igual con ella.
</div>

In [ ]:
# Alternativa sin cuenta: ejecuta esta celda en lugar de la anterior.
from qiskit_ibm_runtime.fake_provider import FakeTorino

backend = FakeTorino()
print(backend.name)

In [ ]:
# Transpila el circuito y optimízalo para ejecutarlo en el computador cuántico elegido
# Paso 2: Transpilar
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)
qc_isa = pm.run(qc)

qc_isa.draw("mpl")

#### Ejecutar

¡Ya podemos ejecutar! Cargaremos Sampler y luego enviaremos el trabajo al *backend*.

In [ ]:
# Carga la primitiva de Runtime y la sesión
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(mode=backend)

O, si estás usando un simulador, puedes descomentar y ejecutar esta celda en su lugar:

In [ ]:
## Carga el sampler para backends
# from qiskit.primitives import BackendSamplerV2

## Carga el simulador Aer y genera un modelo de ruido basado en el backend seleccionado.
# from qiskit_aer import AerSimulator
# from qiskit_aer.noise import NoiseModel

# noise_model = NoiseModel.from_backend(backend)

## Define un simulador con Aer y úsalo en el Sampler.
# backend_sim = AerSimulator(noise_model=noise_model)
# sampler_sim = BackendSamplerV2(backend=backend_sim)

## Como alternativa, carga un backend falso con propiedades genéricas y define un simulador.
## backend_gen = GenericBackendV2(num_qubits=18)
## sampler_gen = BackendSamplerV2(backend=backend_gen)

In [ ]:
job = sampler.run([qc_isa], shots=100)
# job = sampler_sim.run([qc_isa]) # descomenta si quieres ejecutar en un simulador
res = job.result()
counts = res[0].data.meas.get_counts()

#### Post-procesar

In [ ]:
from qiskit.visualization import plot_histogram

print("counts = ", counts)
plot_histogram(counts)

Vemos que la mayoría de los conteos están en `01` o en `10`, lo que significa que cuando un qubit se midió en `0`, el otro estaba en `1`, y viceversa. Esto es coherente con el estado de Bell $\vert \Psi^- \rangle$ que preparamos.

### Experimento 2: usar `Estimator` para medir la energía

Ahora que vimos cómo muestrear un estado cuántico, usemos Estimator para calcular la energía de nuestro estado de Bell $\vert \Psi^- \rangle = \frac{1}{\sqrt{2}}(\vert 01 \rangle - \vert 10 \rangle)$.

#### Mapear

Como recordatorio, la energía del sistema está determinada por la interacción entre los espines ($J$) y el campo magnético externo ($h_x$), según el Hamiltoniano:

$$
H = J Z_1 Z_0 + h_x (X_1 + X_0)
$$

Cada término del Hamiltoniano nos dice cómo contribuye a la energía una cierta combinación de espines. En Qiskit, podemos representar estos términos como operadores de Pauli, que no son más que etiquetas para acciones sencillas sobre los qubits:

* $Z_1 Z_0$ actúa con una $Z$ sobre los dos qubits.
* $X_0$ actúa con una $X$ sobre el qubit 0.
* $X_1$ actúa con una $X$ sobre el qubit 1.

Un `SparsePauliOp` en Qiskit es una forma de guardar una lista de estos operadores de Pauli junto con sus coeficientes numéricos. Estos operadores de Pauli son los observables que queremos que mida el computador cuántico: las cantidades que nos dicen algo sobre el sistema. Con `Estimator`, podemos calcular el valor promedio de cada observable en nuestro estado y combinarlos según los coeficientes del Hamiltoniano para obtener la energía total.

In [ ]:
# Importa las primitivas de IBM Quantum
from qiskit.quantum_info import SparsePauliOp

# Parámetros
J = 1.0  # acoplamiento antiferromagnético (J>0)
hx = -0.5  # intensidad del campo transversal

# 1. Define el Hamiltoniano H = J Z1 Z2 + hx (X1 + X2)
obs = SparsePauliOp.from_list([("ZZ", J), ("XI", hx), ("IX", hx)])

# Prepara el estado
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.x(1)
qc.z(0)

Fíjate en que omitimos la línea `qc.measure_all()` en nuestro código. Esto es porque con `Estimator` no necesitamos indicar dónde medir en el circuito. Solo le decimos qué observables queremos estimar, y Qiskit se encarga de las mediciones internamente.

#### Optimizar

El paso de optimización es igual que antes, con el agregado de asegurarnos de que nuestros observables también estén escritos de una forma que el computador cuántico pueda entender.

In [ ]:
# Transpila el circuito y optimízalo para ejecutarlo en el computador cuántico elegido
# Paso 2: Transpilar
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

target = backend.target
pm = generate_preset_pass_manager(target=target, optimization_level=3)
qc_isa = pm.run(qc)
obs_isa = obs.apply_layout(layout=qc_isa.layout)

qc_isa.draw("mpl")

#### Ejecutar

En el paso de ejecución cargaremos `Estimator` y luego enviaremos al computador cuántico el circuito junto con la lista de observables que queremos que estime.

In [ ]:
# Carga la primitiva de Runtime y la sesión
from qiskit_ibm_runtime import EstimatorV2 as Estimator

estimator = Estimator(mode=backend)

In [ ]:
# Carga el sampler para backends

# noise_model = NoiseModel.from_backend(backend)

# Usa el simulador Aer en Estimator
# estimator_sim = BackendEstimatorV2(backend=backend_sim)

# Como alternativa, carga un backend falso con propiedades genéricas y define un simulador.
# backend_gen = GenericBackendV2(num_qubits=18)
# estimator_gen = BackendEstimatorV2(backend=backend_gen)

In [ ]:
pubs = [(qc_isa, obs_isa)]
job = estimator.run([[qc_isa, obs_isa]])
res = job.result()

# Descomenta las líneas de abajo para ejecutar el trabajo en el simulador Aer con el modelo de ruido del backend real
# job = estimator_sim.run([[qc_isa,obs_isa]])
# res=job.result()

#### Post-procesar

Por último, en el paso de post-procesamiento simplemente imprimimos la energía que `Estimator` calculó internamente.

In [ ]:
print(res[0].data.evs)

¡Esta es la energía de nuestro estado!

## Conclusión

En esta lección aprendimos a preparar un estado cuántico sencillo de dos qubits que representa dos espines que interactúan. Vimos cómo usar `Sampler` para observar la distribución de los resultados de medición y cómo usar `Estimator` para calcular la energía del estado según el Hamiltoniano. En el camino, vimos cómo el Hamiltoniano codifica las interacciones entre espines y los efectos de un campo externo, y cómo distintos estados pueden tener energías distintas.

### Extensión a muchos espines

Hasta ahora solo hemos visto dos espines, un caso lo bastante sencillo para analizarlo a mano. En los sistemas físicos reales, como un imán u otro material complejo, suele haber muchos espines que interactúan. Cuando aumenta el número de espines, el Hamiltoniano se vuelve más complejo y encontrar el estado de menor energía se vuelve mucho más difícil. Aquí es donde los computadores cuánticos pueden ayudar: preparando distintos estados y estimando sus energías, podemos explorar configuraciones de baja energía de forma más eficiente que los computadores clásicos en sistemas grandes.

Una extensión natural de este experimento sería aumentar el número de qubits para representar más espines y ajustar la forma en que se preparan los espines para intentar encontrar el estado de menor energía. Este enfoque es la esencia de los métodos variacionales, que puedes estudiar en el curso [Variational Quantum Algorithms](https://quantum.cloud.ibm.com/learning/es/courses/variational-algorithm-design).

También hay otros enfoques cuánticos para estudiar las energías del estado fundamental que van más allá de las técnicas variacionales. Esos métodos no se tratan aquí, pero se presentan en el curso [Quantum Diagonalization Algorithms](https://quantum.cloud.ibm.com/learning/es/courses/quantum-diagonalization-algorithms) si te interesa aprender más.

#### Objetivo de aprendizaje

Vuelve al principio del Experimento 2 e inténtalo de nuevo con un estado de superposición distinto. ¿Puedes encontrar un estado con una energía todavía más baja que la del que usamos?

© IBM Corp., 2017-2026. Traducción al español para el Qiskit Fall Fest 2026.